# Ejercicio 3 - Índices NDVI, NDWI y cianobacteria

Este cuaderno calcula, para cada combinación oficial lago-fecha, los tres
índices pedidos en el laboratorio: NDVI, NDWI y el índice de cianobacteria.
Depende de los raster crudos que descarga Persona A (`data/raw/rasters/`,
bandas B03, B04, B08, SCL de Sentinel-2 L2A) y del contrato
`manifest_indices.csv` que este notebook entrega a Persona C.

Por defecto el cuaderno solo valida configuración y contrato sin descargar ni
calcular nada remoto, igual que el cuaderno de los ejercicios 1 y 2.

## 1. Decisión del script de cianobacteria

El catálogo de https://custom-scripts.sentinel-hub.com tiene varios scripts
relacionados con agua/algas para Sentinel-2 (NDCI L1C, Se2WaQ, Maximum Peak
Height Bloom Index, APA Script). Se eligió **CyanoLakes Chlorophyll-a L1C
(NDCI)** porque:

- Es el único nombrado explícitamente para cianobacteria, no para calidad de
  agua en general.
- Su fórmula, umbrales y máscara de agua son públicos y auditables (no es
  una caja negra): `src/evalscripts/cyano_ndci_l1c_original.js` es una copia
  textual, sin modificar, del script del catálogo.
- Usa Sentinel-2 **L1C** (reflectancia TOA), el producto para el que fue
  calibrado el polinomio de clorofila-a. Por eso **no se reproduce
  localmente sobre L2A**: se pide directamente a la Process API de
  Sentinel Hub / Copernicus Data Space el resultado numérico de ese script
  (decisión explícita del ejercicio 3.1 del enunciado).

`src/evalscripts/cyano_ndci_l1c_numeric.js` es una adaptación fiel del
script original: mismas fórmulas, mismos umbrales y mismas 9 bandas de
entrada; lo único que cambia es la salida, que deja de ser un color RGB de
visualización y pasa a ser un valor numérico FLOAT32 (`chl`), con una banda
`dataMask` que marca como válido únicamente lo que el propio script clasifica
como agua. Esto es necesario porque, como señala el enunciado, "una imagen
RGB coloreada sirve para visualizar, pero no para recuperar de forma válida
el índice promedio".

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Abra Jupyter desde la raíz de Laboratorio 4')

sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import display

from src import indices


In [ ]:
from src.config import (
    CYANO_SCRIPT,
    ESCENAS_OFICIALES,
    INDICES,
    RESOLUCION_OBJETIVO_M,
)
from src.indices import (
    compute_indices_batch,
    compute_indices_for_scene,
    prepare_indices_repository,
    request_cyano_layer,
    select_scenes,
    validate_manifest_indices,
)

print(f'Raíz del proyecto: {ROOT}')
print(f'Índices a calcular: {INDICES}')
print(f'Resolución objetivo: {RESOLUCION_OBJETIVO_M} m')


## 2. Preparación local y contrato con Persona C

`manifest_indices.csv` nace con 66 filas (22 escenas oficiales x 3 índices)
en estado `pendiente_calculo`. Cada fila se completa únicamente cuando el
índice correspondiente se calcula y exporta con éxito.

In [ ]:
summary = prepare_indices_repository()
rows = validate_manifest_indices()
manifest = pd.DataFrame(rows)

print(f"Filas del manifiesto de índices: {summary['rows']}")
display(manifest.groupby(['indice', 'quality_flag']).size().rename('n').reset_index())


## 3. Fórmulas documentadas

Las mismas fórmulas se usan en `src/indices.py`; aquí solo se muestran para
que queden visibles junto a los resultados.

In [ ]:
formulas = pd.Series(
    {
        'NDVI': 'NDVI = (B08 - B04) / (B08 + B04)',
        'NDWI': 'NDWI = (B03 - B08) / (B03 + B08)',
        'Cianobacteria (NDCI)': CYANO_SCRIPT['formula_ndci'],
        'Cianobacteria (chl-a)': CYANO_SCRIPT['formula_chl'],
        'Máscara de agua del script': CYANO_SCRIPT['mascara'],
        'Unidad cianobacteria': CYANO_SCRIPT['unidad'],
        'Script fuente': CYANO_SCRIPT['url_fuente'],
    },
    name='definición',
)
formulas.to_frame()


## 4. Escena demostrativa (una fecha)

Requiere que Persona A ya haya descargado B03, B04, B08 y SCL para esta
fecha (`python src/adquisicion.py download --lago amatitlan --fecha
2025-01-28`). Si el raster crudo de cianobacteria aún no existe localmente,
`EJECUTAR_CYANO_REMOTO=True` lo pide a la Process API de Sentinel Hub
(requiere `SENTINEL_HUB_CLIENT_ID`/`SENTINEL_HUB_CLIENT_SECRET` en `.env`).

Ambas banderas están desactivadas por defecto para que el cuaderno sea
reproducible sin credenciales ni raster descargados.

In [ ]:
EJECUTAR_DEMO = False
EJECUTAR_CYANO_REMOTO = False

demo_scene = select_scenes('amatitlan', '2025-01-28')[0]

if EJECUTAR_DEMO:
    if EJECUTAR_CYANO_REMOTO:
        cyano_path = request_cyano_layer(demo_scene)
        print(f'Cianobacteria descargada: {cyano_path}')
    outputs = compute_indices_for_scene(demo_scene, fetch_cyano_remote=False)
    for indice, path in outputs.items():
        print(f'- {indice}: {path}')
else:
    print('Demo omitida. Requiere raster crudos de Persona A para', demo_scene.lago, demo_scene.fecha)
    print('Cambie EJECUTAR_DEMO=True (y EJECUTAR_CYANO_REMOTO=True si falta el raster de cianobacteria).')


## 5. Control numérico y visual de la escena demostrativa

Solo aplica si la celda anterior se ejecutó con éxito: compara los tres
índices exportados contra el manifiesto y muestra un mapa rápido de cada
uno.

In [ ]:
if EJECUTAR_DEMO:
    import matplotlib.pyplot as plt
    import rasterio

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, indice in zip(axes, INDICES):
        path = outputs[indice]
        with rasterio.open(path) as dataset:
            array = dataset.read(1)
        im = ax.imshow(array)
        ax.set_title(indice)
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.suptitle(f"{demo_scene.lago} {demo_scene.fecha}")
    plt.tight_layout()
    plt.show()
else:
    print('Sin datos de la demo todavía.')


## 6. Cálculo del lote de 22 escenas

Solo después de validar la escena demostrativa. Requiere que Persona A haya
descargado las 22 escenas oficiales y, si no se han pedido antes, pide a
Sentinel Hub el raster crudo de cianobacteria de cada una
(`fetch_cyano_remote=True`).

In [ ]:
EJECUTAR_LOTE = False

if EJECUTAR_LOTE:
    resultados = compute_indices_batch(
        list(ESCENAS_OFICIALES), fetch_cyano_remote=True, confirm_batch=True
    )
    print(f'Escenas procesadas: {len(resultados)}')
else:
    print('Lote omitido. Cambie EJECUTAR_LOTE=True para calcular las 22 escenas.')


## 7. Tabla de control de calidad

Mínimos, máximos, promedio y cobertura válida por producto, leídos
directamente de `manifest_indices.csv` (fuente única para Persona C, según
el contrato de la planificación de avances).

In [ ]:
rows = validate_manifest_indices()
manifest = pd.DataFrame(rows)
numeric_cols = ['pixeles_validos', 'pixeles_lago', 'cobertura_valida_pct']
manifest = manifest.assign(**{col: pd.to_numeric(manifest[col], errors='coerce') for col in numeric_cols})

resumen_calidad = (
    manifest.groupby('indice')[numeric_cols]
    .agg(['count', 'mean', 'min', 'max'])
)
display(resumen_calidad)
display(manifest[manifest['quality_flag'] == 'cobertura_parcial_oficial'])


## 8. Resultado del ejercicio 3 y entrega a Persona C

Cuando el lote de 22 escenas se ejecuta con éxito, `manifest_indices.csv`
contiene una fila por lago, fecha e índice con: ruta del GeoTIFF de una sola
banda `float32`, método, versión de fórmula, unidad, CRS, resolución,
píxeles válidos y cobertura. Persona C debe leer únicamente este archivo
(no rutas propias) para calcular el promedio temporal de cianobacteria del
ejercicio 4.

**Limitación explícita:** mientras no exista el contorno oficial (GeoJSON)
del lago, la máscara espacial usada para NDVI/NDWI es la clase "agua" de la
banda SCL dentro del bbox de consulta, no un polígono fijo del lago. Esto
excluye nubes/sombras de cada fecha, pero debe reemplazarse por la
intersección con el GeoJSON oficial en cuanto esté disponible. La escena
`amatitlan 2026-02-07` conserva su advertencia de cobertura parcial oficial
(~57.1 %) y no debe compararse como si tuviera la misma confiabilidad que
una escena completa.